# Supply Chain Disruption & Cost Model Training

This notebook trains the ML artifacts used by the Supply Chain GenAI Agent. It creates the XGBoost disruption classifier, shipping-cost regressors, feature configuration, evaluation reports, and mitigation playbook used later by the RAG/LangGraph inference agent.

It creates these downloadable artifacts in Kaggle's `/kaggle/working` directory:

- `supply_chain_agent_training_outputs/models/disruption_classifier.pkl`
- `supply_chain_agent_training_outputs/models/disruption_classifier_base_xgb.pkl`
- `supply_chain_agent_training_outputs/models/disruption_classifier_calibrated.pkl` (when calibration is enabled)
- `supply_chain_agent_training_outputs/models/active_exception_detector.pkl`
- `supply_chain_agent_training_outputs/models/cost_predictor.pkl`
- `supply_chain_agent_training_outputs/models/cost_predictor_enhanced.pkl` (extra improved cost model)
- `supply_chain_agent_training_outputs/knowledge_base/mitigation_playbook.txt`
- `supply_chain_agent_training_outputs/reports/metrics.json`
- `supply_chain_agent_training_outputs/reports/metrics_summary.csv`
- `supply_chain_agent_training_outputs/reports/data_quality_report.json`
- `supply_chain_model_artifacts.zip`

No OpenAI key, LangGraph, FAISS, or LangChain setup is needed for this training notebook. It only trains the local ML models and creates the text knowledge base. Build the FAISS index separately with the GenAI agent: `python supply_chain_genai_agent_groq_hf.py index`.

## Kaggle setup

1. Create a Kaggle Notebook.
2. Add your CSV as an input dataset.
3. Import or upload this `.ipynb` file.
4. Run all cells.
5. Download `supply_chain_model_artifacts.zip` from the right-side Output panel.

In [ ]:
# Cell 1: Imports and runtime configuration
# Fixes made in this version:
# 1. Uses a clean 60/20/20 train/validation/final-test split.
# 2. Uses validation only for early stopping, calibration, and threshold tuning.
# 3. Evaluates metrics once on the untouched final test set.
# 4. Computes lane_disruption_rate from the training split only.
# 5. Removes SMOTE by default for the one-hot feature matrix and selects the classifier weighting mode on validation only.
# 6. Adds classifier baselines, calibration diagnostics, cost baselines, WAPE, RMSLE, and error quantiles.

import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

import json
import math
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
try:
    from sklearn.frozen import FrozenEstimator
except Exception:  # pragma: no cover - compatibility fallback
    FrozenEstimator = None
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_log_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [ ]:
# Cell 2: Locate the input CSV and configure output paths

RANDOM_STATE = 42
N_JOBS = min(os.cpu_count() or 1, 4)
CSV_FILENAME = "global_supply_chain_disruption_v1.csv"

search_roots = []
for candidate in [Path("/kaggle/input"), Path.cwd(), Path("/mnt/data")]:
    if candidate.exists():
        search_roots.append(candidate)

csv_matches = []
for root in search_roots:
    csv_matches.extend(root.rglob(CSV_FILENAME))

# Fallback: if the exact file name is different in Kaggle, use the first CSV found.
if not csv_matches:
    for root in search_roots:
        csv_matches.extend(root.rglob("*.csv"))

if not csv_matches:
    raise FileNotFoundError(
        "Could not find a CSV file. In Kaggle, attach the dataset containing "
        f"{CSV_FILENAME}, then rerun the notebook."
    )

DATA_PATH = csv_matches[0]
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_DIR = WORKING_DIR / "supply_chain_agent_training_outputs"
MODELS_DIR = OUTPUT_DIR / "models"
KB_DIR = OUTPUT_DIR / "knowledge_base"
REPORTS_DIR = OUTPUT_DIR / "reports"

# Clean old generated outputs so every run is reproducible.
if OUTPUT_DIR.exists():
    for child in OUTPUT_DIR.iterdir():
        if child.is_file():
            child.unlink()
        elif child.is_dir() and child.name in {"models", "knowledge_base", "reports"}:
            shutil.rmtree(child)
for directory in [OUTPUT_DIR, MODELS_DIR, KB_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Data path : {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Using N_JOBS={N_JOBS}")


In [ ]:
# Cell 3: Domain constants, feature lists, and training switches

DISRUPTION_PROB_THRESHOLD_SEED = 0.35
USE_LEAKAGE_SAFE_CLASSIFIER = True
USE_SMOTE = False  # Avoid synthetic fractional one-hot categories; class weighting is used instead.
CALIBRATE_CLASSIFIER = True
CALIBRATION_METHOD = "isotonic"
OPTIMIZE_THRESHOLD = True
THRESHOLD_MIN_PRECISION = 0.20
THRESHOLD_MIN_RECALL = 0.20
THRESHOLD_MAX_ALERT_RATE = 0.25
TRAIN_ENHANCED_COST_MODEL = True

ROUTE_RISK_MAP = {
    "Suez": 0.91,
    "Pacific": 0.74,
    "Atlantic": 0.58,
    "Intra-Asia": 0.45,
    "Commodity": 0.28,
}
PRODUCT_CRITICALITY_MAP = {
    "Pharmaceuticals": 1.00,
    "Perishable Foods": 0.95,
    "Semiconductors": 0.70,
    "Consumer Electronics": 0.55,
    "Auto Parts": 0.50,
    "Raw Materials": 0.25,
    "Textiles": 0.15,
}
DISRUPTION_LABEL_MAP = {
    "No Disruption": 0,
    "Port Congestion": 1,
    "Geopolitical Conflict (Route Diversion)": 2,
    "Severe Weather (Typhoon/Storm)": 3,
}
NO_DISRUPTION_VALUES = {"", "none", "nan", "no disruption", "no_disruption", "no-disruption", "null"}

TEMPORAL_FEATURES = [
    "order_month",
    "order_quarter",
    "is_typhoon_season",
    "is_peak_shipping",
    "is_lunar_new_year_window",
]
LANE_FEATURES = ["lane_disruption_rate"]

CLASSIFIER_FEATURES_DEMO = [
    "Geopolitical_Risk_Index", "Weather_Severity_Index", "Inflation_Rate_Pct",
    "Shipping_Cost_USD", "Scheduled_Lead_Time_Days",
    "lead_time_buffer", "delay_ratio", "composite_risk_score",
    "geo_weather_interaction", "route_risk_score", "product_criticality",
    "log_shipping_cost", "cost_percentile",
    "mode_Sea", "mode_Air",
    "route_Suez", "route_Pacific", "route_Atlantic",
    "route_Intra-Asia", "route_Commodity",
    "product_Pharmaceuticals", "product_Perishable Foods",
    "product_Semiconductors", "product_Auto Parts",
]
CLASSIFIER_FEATURES_LEAKAGE_SAFE = [
    f for f in CLASSIFIER_FEATURES_DEMO
    if f not in {"delay_ratio", "Shipping_Cost_USD", "log_shipping_cost", "cost_percentile"}
]
CLASSIFIER_FEATURES = CLASSIFIER_FEATURES_LEAKAGE_SAFE + TEMPORAL_FEATURES + LANE_FEATURES

ACTIVE_EXCEPTION_FEATURES = [
    "observed_delay_days",
    "raw_lead_time_delta_days",
    "delivery_status_late",
    "delay_ratio",
    "Scheduled_Lead_Time_Days",
    "Actual_Lead_Time_Days",
    "lead_time_buffer",
    "mode_Sea",
    "mode_Air",
    "route_Suez",
    "route_Pacific",
    "route_Atlantic",
    "route_Intra-Asia",
    "route_Commodity",
]

COST_MODEL_FEATURES_AGENT_COMPATIBLE = [
    "Geopolitical_Risk_Index", "Weather_Severity_Index",
    "Scheduled_Lead_Time_Days", "route_risk_score", "product_criticality",
    "mode_Sea", "mode_Air", "geo_weather_interaction",
    "route_Suez", "route_Pacific", "route_Atlantic",
]
COST_MODEL_FEATURES_ENHANCED = [
    "Geopolitical_Risk_Index",
    "Weather_Severity_Index",
    "Scheduled_Lead_Time_Days",
    "Base_Lead_Time_Days",
    "lead_time_buffer",
    "Order_Weight_Kg",
    "route_risk_score",
    "product_criticality",
    "geo_weather_interaction",
    "mode_Sea",
    "mode_Air",
    "route_Suez",
    "route_Pacific",
    "route_Atlantic",
    "route_Intra-Asia",
    "route_Commodity",
    "product_Pharmaceuticals",
    "product_Perishable Foods",
    "product_Semiconductors",
    "product_Auto Parts",
    "product_Consumer Electronics",
    "product_Raw Materials",
    "product_Textiles",
] + TEMPORAL_FEATURES

ALL_EXPECTED_FEATURES = sorted(
    set(CLASSIFIER_FEATURES_DEMO)
    | set(CLASSIFIER_FEATURES_LEAKAGE_SAFE)
    | set(CLASSIFIER_FEATURES)
    | set(COST_MODEL_FEATURES_AGENT_COMPATIBLE)
    | set(COST_MODEL_FEATURES_ENHANCED)
    | set(ACTIVE_EXCEPTION_FEATURES)
)

print(f"Classifier features  : {len(CLASSIFIER_FEATURES)}")
print(f"Cost features        : agent={len(COST_MODEL_FEATURES_AGENT_COMPATIBLE)}, enhanced={len(COST_MODEL_FEATURES_ENHANCED)}")

In [ ]:
# Cell 4: Data preparation, validation, and feature engineering

def normalize_disruption_event(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text.lower() in NO_DISRUPTION_VALUES:
        return np.nan
    canonical = {key.lower(): key for key in DISRUPTION_LABEL_MAP}
    return canonical.get(text.lower(), text)


def ensure_feature_columns(df: pd.DataFrame, features: Iterable[str]) -> pd.DataFrame:
    df = df.copy()
    for col in features:
        if col not in df.columns:
            df[col] = 0
    return df


def validate_raw_columns(df: pd.DataFrame) -> None:
    required = {
        "Order_ID", "Order_Date", "Origin_City", "Destination_City",
        "Route_Type", "Transportation_Mode", "Product_Category",
        "Base_Lead_Time_Days", "Scheduled_Lead_Time_Days", "Actual_Lead_Time_Days",
        "Delay_Days", "Delivery_Status", "Disruption_Event",
        "Geopolitical_Risk_Index", "Weather_Severity_Index", "Inflation_Rate_Pct",
        "Shipping_Cost_USD", "Order_Weight_Kg", "Mitigation_Action_Taken",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Input CSV is missing required columns: {missing}")


def add_delay_and_label_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["raw_lead_time_delta_days"] = df["Actual_Lead_Time_Days"] - df["Scheduled_Lead_Time_Days"]
    df["observed_delay_days"] = df["raw_lead_time_delta_days"].clip(lower=0)
    df["delay_days_delta_vs_raw_delta"] = df["Delay_Days"] - df["raw_lead_time_delta_days"]
    df["delay_days_delta_vs_policy"] = df["Delay_Days"] - df["observed_delay_days"]
    df["delay_matches_late_days_policy"] = (df["delay_days_delta_vs_policy"].abs() < 1e-9).astype(int)
    status = df["Delivery_Status"].astype(str).str.strip().str.lower()
    df["delivery_status_late"] = (~status.isin(["on time", "ontime"])).astype(int)
    df["is_late"] = (
        (df["observed_delay_days"] > 0)
        | (df["Delay_Days"] > 0)
        | (df["delivery_status_late"] == 1)
    ).astype(int)
    df["is_event_disruption"] = df["Disruption_Event"].notna().astype(int)
    df["is_disrupted"] = df["is_event_disruption"]
    df["is_active_exception"] = ((df["is_event_disruption"] == 1) | (df["is_late"] == 1)).astype(int)
    df["event_but_on_time"] = ((df["is_event_disruption"] == 1) & (df["is_late"] == 0)).astype(int)
    df["late_without_event"] = ((df["is_late"] == 1) & (df["is_event_disruption"] == 0)).astype(int)
    return df


def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    order_ts = pd.to_datetime(df["Order_Date"], errors="coerce")
    df["order_month"] = order_ts.dt.month.fillna(0).astype(int)
    df["order_quarter"] = order_ts.dt.quarter.fillna(0).astype(int)
    df["is_typhoon_season"] = df["order_month"].isin([8, 9, 10]).astype(int)
    df["is_peak_shipping"] = df["order_month"].isin([10, 11, 12]).astype(int)
    df["is_lunar_new_year_window"] = df["order_month"].isin([1, 2]).astype(int)
    return df


def add_non_target_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["lead_time_buffer"] = df["Scheduled_Lead_Time_Days"] - df["Base_Lead_Time_Days"]
    df["delay_ratio"] = df["observed_delay_days"] / (df["Scheduled_Lead_Time_Days"] + 1e-6)
    df["geo_weather_interaction"] = df["Geopolitical_Risk_Index"] * df["Weather_Severity_Index"]
    df["inflation_risk_component"] = df["Inflation_Rate_Pct"].clip(lower=0) / 10
    df["composite_risk_score"] = (
        df["Geopolitical_Risk_Index"] * 0.50
        + (df["Weather_Severity_Index"] / 10) * 0.30
        + df["inflation_risk_component"] * 0.20
    )
    df["route_risk_score"] = df["Route_Type"].map(ROUTE_RISK_MAP).fillna(0.5)
    df["product_criticality"] = df["Product_Category"].map(PRODUCT_CRITICALITY_MAP).fillna(0.5)
    df["log_shipping_cost"] = np.log1p(df["Shipping_Cost_USD"])
    # This is still created for legacy/demo feature compatibility but is not used by the leakage-safe classifier.
    df["cost_percentile"] = df.groupby("Route_Type")["Shipping_Cost_USD"].rank(pct=True)
    df["air_viable"] = ((df["product_criticality"] >= 0.5) & (df["observed_delay_days"] >= 5)).astype(int)
    df["lane"] = df["Origin_City"].astype(str).str.strip() + " -> " + df["Destination_City"].astype(str).str.strip()
    df = add_temporal_features(df)
    for col, prefix in [
        ("Transportation_Mode", "mode"),
        ("Route_Type", "route"),
        ("Product_Category", "product"),
    ]:
        dummies = pd.get_dummies(df[col], prefix=prefix, dtype=int)
        df = pd.concat([df, dummies], axis=1)
    df = ensure_feature_columns(df, ALL_EXPECTED_FEATURES)
    return df


def load_and_engineer(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    validate_raw_columns(df)
    df["Disruption_Event"] = df["Disruption_Event"].apply(normalize_disruption_event)
    df = add_delay_and_label_features(df)
    df["disruption_label"] = (
        df["Disruption_Event"].fillna("No Disruption").map(DISRUPTION_LABEL_MAP).fillna(-1).astype(int)
    )
    unknown = sorted(df.loc[df["disruption_label"] == -1, "Disruption_Event"].dropna().astype(str).unique())
    if unknown:
        raise ValueError(f"Unknown Disruption_Event labels found: {unknown}")
    df = add_non_target_features(df)
    return df


def add_train_only_lane_rate(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_col: str = "is_disrupted",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, float], float]:
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()
    lane_rate = train_df.groupby("lane")[target_col].mean().astype(float)
    global_rate = float(train_df[target_col].mean())
    for part in [train_df, val_df, test_df]:
        part["lane_disruption_rate"] = part["lane"].map(lane_rate).fillna(global_rate).astype(float)
    lane_map = {str(k): float(v) for k, v in lane_rate.items()}
    lane_map["__global__"] = global_rate
    return train_df, val_df, test_df, lane_map, global_rate


def build_data_quality_report(df: pd.DataFrame) -> Dict[str, object]:
    p99_cost = float(df["Shipping_Cost_USD"].quantile(0.99))
    return {
        "row_count": int(len(df)),
        "column_count": int(df.shape[1]),
        "duplicate_order_id_count": int(df["Order_ID"].duplicated().sum()),
        "raw_delay_mismatch_rows": int((df["Delay_Days"] != df["raw_lead_time_delta_days"]).sum()),
        "delay_policy_mismatch_rows": int((df["delay_matches_late_days_policy"] == 0).sum()),
        "event_but_on_time_rows": int(df["event_but_on_time"].sum()),
        "late_without_event_rows": int(df["late_without_event"].sum()),
        "negative_inflation_rows": int((df["Inflation_Rate_Pct"] < 0).sum()),
        "shipping_cost_p99_usd": p99_cost,
        "shipping_cost_max_usd": float(df["Shipping_Cost_USD"].max()),
        "event_disruption_rate": float(df["is_event_disruption"].mean()),
        "late_delivery_rate": float(df["is_late"].mean()),
        "active_exception_rate": float(df["is_active_exception"].mean()),
        "note": "Delay_Days is interpreted as max(Actual_Lead_Time_Days - Scheduled_Lead_Time_Days, 0), not as raw arrival variance.",
    }

In [ ]:
# Cell 5: Load and inspect the dataset

df = load_and_engineer(DATA_PATH)
data_quality_report = build_data_quality_report(df)

print(f"Rows: {len(df):,}")
print(f"Columns after feature engineering: {df.shape[1]:,}")
print(f"Event disruption rate: {df['is_event_disruption'].mean():.2%}")
print(f"Late delivery rate    : {df['is_late'].mean():.2%}")
print(f"Active exception rate : {df['is_active_exception'].mean():.2%}")
print("\nData quality report:")
print(pd.DataFrame.from_dict(data_quality_report, orient="index", columns=["value"]).to_string())
print("\nDisruption distribution:")
print(df["Disruption_Event"].fillna("No Disruption").value_counts(dropna=False).to_string())

In [ ]:
# Cell 6: Training helpers with clean validation/final-test separation

def classification_metrics_dict(y_true, y_prob, threshold: float) -> Dict[str, object]:
    y_true = pd.Series(y_true).astype(int).to_numpy()
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    positive_rate = float(np.mean(y_true))
    alert_rate = float(np.mean(y_pred))
    precision = float(precision_score(y_true, y_pred, zero_division=0))
    return {
        "threshold": float(threshold),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": precision,
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "positive_rate": positive_rate,
        "alert_rate": alert_rate,
        "false_alerts_per_true_alert": None if tp == 0 else float(fp / tp),
        "precision_lift_vs_prevalence": None if positive_rate == 0 else float(precision / positive_rate),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def classification_baselines(y_train, y_test) -> Dict[str, object]:
    y_train = pd.Series(y_train).astype(int).to_numpy()
    y_test = pd.Series(y_test).astype(int).to_numpy()
    train_rate = float(np.mean(y_train))
    test_rate = float(np.mean(y_test))
    constant_prob = np.full_like(y_test, fill_value=train_rate, dtype=float)
    all_negative = np.zeros_like(y_test, dtype=int)
    return {
        "train_positive_rate": train_rate,
        "test_positive_rate": test_rate,
        "always_negative_accuracy": float(accuracy_score(y_test, all_negative)),
        "always_negative_precision": 0.0,
        "always_negative_recall": 0.0,
        "always_negative_f1": 0.0,
        "prevalence_pr_auc_baseline": test_rate,
        "constant_train_rate_brier_score": float(brier_score_loss(y_test, constant_prob)),
    }


def threshold_sweep_frame(y_true, y_prob, thresholds=None) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.arange(0.01, 1.00, 0.01)
    rows = []
    y_true = pd.Series(y_true).astype(int).to_numpy()
    y_prob = np.asarray(y_prob, dtype=float)
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        rows.append({
            "threshold": float(t),
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "alert_rate": float(y_pred.mean()),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        })
    return pd.DataFrame(rows)


def find_operational_threshold(y_true, y_prob) -> Tuple[float, str, pd.DataFrame]:
    sweep = threshold_sweep_frame(y_true, y_prob)
    feasible = sweep[
        (sweep["precision"] >= THRESHOLD_MIN_PRECISION)
        & (sweep["recall"] >= THRESHOLD_MIN_RECALL)
        & (sweep["alert_rate"] <= THRESHOLD_MAX_ALERT_RATE)
    ]
    if not feasible.empty:
        best = feasible.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]
        return float(best["threshold"]), "met_precision_recall_and_alert_rate_constraints_on_validation", sweep
    capped = sweep[sweep["alert_rate"] <= THRESHOLD_MAX_ALERT_RATE]
    if not capped.empty:
        best = capped.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
        return float(best["threshold"]), "constraints_not_fully_reachable_used_best_f1_under_alert_cap_on_validation", sweep
    best = sweep.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]
    return float(best["threshold"]), "alert_cap_not_reachable_used_best_validation_f1", sweep


def cost_metrics_dict(y_true, y_pred, y_train=None) -> Dict[str, object]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    abs_error = np.abs(y_true - y_pred)
    pct_error = abs_error / np.maximum(np.abs(y_true), 1e-6)
    metrics = {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "median_absolute_error": float(np.median(abs_error)),
        "p90_absolute_error": float(np.quantile(abs_error, 0.90)),
        "p95_absolute_error": float(np.quantile(abs_error, 0.95)),
        "mape_pct": float(mean_absolute_percentage_error(y_true, y_pred) * 100),
        "median_ape_pct": float(np.median(pct_error) * 100),
        "wape_pct": float(abs_error.sum() / max(np.abs(y_true).sum(), 1e-6) * 100),
        "rmsle": float(np.sqrt(mean_squared_log_error(np.maximum(y_true, 0), y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }
    if y_train is not None:
        y_train = np.asarray(y_train, dtype=float)
        mean_pred = np.full_like(y_true, y_train.mean(), dtype=float)
        median_pred = np.full_like(y_true, np.median(y_train), dtype=float)
        metrics.update({
            "train_mean_baseline_mae": float(mean_absolute_error(y_true, mean_pred)),
            "train_median_baseline_mae": float(mean_absolute_error(y_true, median_pred)),
            "train_mean_baseline_wape_pct": float(np.abs(y_true - mean_pred).sum() / max(np.abs(y_true).sum(), 1e-6) * 100),
            "train_median_baseline_wape_pct": float(np.abs(y_true - median_pred).sum() / max(np.abs(y_true).sum(), 1e-6) * 100),
            "train_mean_baseline_rmsle": float(np.sqrt(mean_squared_log_error(np.maximum(y_true, 0), np.maximum(mean_pred, 0)))),
            "train_median_baseline_rmsle": float(np.sqrt(mean_squared_log_error(np.maximum(y_true, 0), np.maximum(median_pred, 0)))),
        })
    return metrics


def split_indices_for_classification(y: pd.Series, test_size=0.20, val_size=0.20):
    idx = np.arange(len(y))
    trainval_idx, test_idx = train_test_split(
        idx, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )
    relative_val_size = val_size / (1.0 - test_size)
    train_idx, val_idx = train_test_split(
        trainval_idx,
        test_size=relative_val_size,
        random_state=RANDOM_STATE,
        stratify=y.iloc[trainval_idx],
    )
    return train_idx, val_idx, test_idx


def split_indices_for_regression(n: int, test_size=0.20, val_size=0.20):
    idx = np.arange(n)
    trainval_idx, test_idx = train_test_split(idx, test_size=test_size, random_state=RANDOM_STATE)
    relative_val_size = val_size / (1.0 - test_size)
    train_idx, val_idx = train_test_split(trainval_idx, test_size=relative_val_size, random_state=RANDOM_STATE)
    return train_idx, val_idx, test_idx

# Disruption classifier: validation-only early stopping/calibration/threshold tuning, final-test evaluation

def train_disruption_classifier(data: pd.DataFrame, features: List[str]):
    """Train the pre-event disruption classifier with no final-test leakage.

    Two XGBoost weighting candidates are trained on the training split only:
    unweighted and balanced-scale-pos-weight. Both are early-stopped, calibrated,
    threshold-tuned, and selected using validation metrics only. The untouched
    final test split is evaluated once for the selected candidate.
    """
    y = data["is_disrupted"].copy()
    train_idx, val_idx, test_idx = split_indices_for_classification(y)
    train_df, val_df, test_df, lane_map, lane_global = add_train_only_lane_rate(
        data.iloc[train_idx], data.iloc[val_idx], data.iloc[test_idx], target_col="is_disrupted"
    )
    X_train, y_train = train_df[features].copy(), train_df["is_disrupted"].copy()
    X_val, y_val = val_df[features].copy(), val_df["is_disrupted"].copy()
    X_test, y_test = test_df[features].copy(), test_df["is_disrupted"].copy()

    balanced_scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    candidate_specs = [
        {"candidate": "unweighted_no_smote", "scale_pos_weight": 1.0},
        {"candidate": "balanced_scale_pos_no_smote", "scale_pos_weight": float(balanced_scale_pos)},
    ]

    candidate_objects = []
    candidate_rows = []

    for spec in candidate_specs:
        base_model = XGBClassifier(
            n_estimators=1000,
            max_depth=4,
            learning_rate=0.02,
            min_child_weight=5,
            gamma=1.0,
            scale_pos_weight=float(spec["scale_pos_weight"]),
            subsample=0.80,
            colsample_bytree=0.70,
            reg_alpha=0.1,
            reg_lambda=2.0,
            eval_metric="aucpr",
            tree_method="hist",
            early_stopping_rounds=50,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
        base_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        raw_val_prob = base_model.predict_proba(X_val)[:, 1]
        raw_test_prob = base_model.predict_proba(X_test)[:, 1]

        calibrated_model = None
        if CALIBRATE_CLASSIFIER:
            if FrozenEstimator is not None:
                calibrated_model = CalibratedClassifierCV(FrozenEstimator(base_model), method=CALIBRATION_METHOD)
                calibrated_model.fit(X_val, y_val)
            else:  # Compatibility fallback for older sklearn versions.
                calibrated_model = CalibratedClassifierCV(base_model, method=CALIBRATION_METHOD, cv="prefit")
                calibrated_model.fit(X_val, y_val)
            val_prob = calibrated_model.predict_proba(X_val)[:, 1]
            test_prob = calibrated_model.predict_proba(X_test)[:, 1]
        else:
            val_prob = raw_val_prob
            test_prob = raw_test_prob

        if OPTIMIZE_THRESHOLD:
            threshold, threshold_status, val_sweep = find_operational_threshold(y_val, val_prob)
        else:
            threshold, threshold_status, val_sweep = DISRUPTION_PROB_THRESHOLD_SEED, "fixed_threshold"

        val_metrics = classification_metrics_dict(y_val, val_prob, threshold)
        raw_val_metrics = classification_metrics_dict(y_val, raw_val_prob, threshold)
        row = {
            "candidate": spec["candidate"],
            "scale_pos_weight": float(spec["scale_pos_weight"]),
            "threshold": float(threshold),
            "threshold_status": threshold_status,
            "validation_roc_auc": val_metrics["roc_auc"],
            "validation_pr_auc": val_metrics["pr_auc"],
            "validation_precision": val_metrics["precision"],
            "validation_recall": val_metrics["recall"],
            "validation_f1": val_metrics["f1"],
            "validation_accuracy": val_metrics["accuracy"],
            "validation_alert_rate": val_metrics["alert_rate"],
            "validation_brier_score": val_metrics["brier_score"],
            "raw_validation_roc_auc": raw_val_metrics["roc_auc"],
            "raw_validation_pr_auc": raw_val_metrics["pr_auc"],
            "raw_validation_brier_score": raw_val_metrics["brier_score"],
            "best_iteration": None if getattr(base_model, "best_iteration", None) is None else int(base_model.best_iteration),
        }
        candidate_rows.append(row)
        candidate_objects.append({
            "spec": spec,
            "base_model": base_model,
            "calibrated_model": calibrated_model,
            "raw_test_prob": raw_test_prob,
            "test_prob": test_prob,
            "val_prob": val_prob,
            "threshold": threshold,
            "threshold_status": threshold_status,
            "val_sweep": val_sweep,
            "val_metrics": val_metrics,
            "row": row,
        })

    candidate_df = pd.DataFrame(candidate_rows).sort_values(
        ["validation_f1", "validation_precision", "validation_recall", "validation_pr_auc"],
        ascending=False,
    )
    candidate_df.to_csv(REPORTS_DIR / "classifier_candidate_selection_validation_only.csv", index=False)
    selected_name = str(candidate_df.iloc[0]["candidate"])
    selected = next(obj for obj in candidate_objects if obj["spec"]["candidate"] == selected_name)

    threshold = float(selected["threshold"])
    threshold_status = selected["threshold_status"]
    test_prob = selected["test_prob"]
    raw_test_prob = selected["raw_test_prob"]
    base_model = selected["base_model"]
    calibrated_model = selected["calibrated_model"]
    val_metrics_at_threshold = selected["val_metrics"]

    selected["val_sweep"].to_csv(REPORTS_DIR / "classifier_validation_threshold_sweep.csv", index=False)
    test_sweep = threshold_sweep_frame(y_test, test_prob)
    test_sweep.to_csv(REPORTS_DIR / "classifier_test_threshold_sweep_diagnostic_not_used_for_tuning.csv", index=False)

    final_metrics = classification_metrics_dict(y_test, test_prob, threshold)
    raw_test_metrics = classification_metrics_dict(y_test, raw_test_prob, threshold)
    baselines = classification_baselines(y_train, y_test)
    final_metrics.update({
        "model_file": "disruption_classifier.pkl",
        "calibrated_model_file": "disruption_classifier_calibrated.pkl" if CALIBRATE_CLASSIFIER else None,
        "base_model_file": "disruption_classifier_base_xgb.pkl",
        "target": "is_event_disruption",
        "mode": "leakage_safe_pre_event_classifier",
        "selected_classifier_candidate": selected_name,
        "candidate_selection_metric": "validation_f1_then_precision_recall_pr_auc",
        "candidate_selection_tuned_on": "validation",
        "candidate_selection_report_file": "classifier_candidate_selection_validation_only.csv",
        "classifier_candidate_validation_summary": candidate_rows,
        "feature_count": len(features),
        "smote_applied": bool(USE_SMOTE),
        "scale_pos_weight": float(selected["spec"]["scale_pos_weight"]),
        "calibrated": bool(CALIBRATE_CLASSIFIER),
        "calibration_method": CALIBRATION_METHOD if CALIBRATE_CLASSIFIER else None,
        "threshold_tuned_on": "validation",
        "threshold_status": threshold_status,
        "threshold_min_precision": float(THRESHOLD_MIN_PRECISION),
        "threshold_min_recall": float(THRESHOLD_MIN_RECALL),
        "threshold_max_alert_rate": float(THRESHOLD_MAX_ALERT_RATE),
        "best_iteration": None if getattr(base_model, "best_iteration", None) is None else int(base_model.best_iteration),
        "split_train_rows": int(len(train_df)),
        "split_validation_rows": int(len(val_df)),
        "split_final_test_rows": int(len(test_df)),
        "train_positive_rate": float(y_train.mean()),
        "validation_positive_rate": float(y_val.mean()),
        "test_positive_rate": float(y_test.mean()),
        "validation_metrics_at_chosen_threshold": val_metrics_at_threshold,
        "raw_uncalibrated_test_roc_auc": raw_test_metrics["roc_auc"],
        "raw_uncalibrated_test_pr_auc": raw_test_metrics["pr_auc"],
        "raw_uncalibrated_test_brier_score": raw_test_metrics["brier_score"],
        "baselines": baselines,
    })

    importance = pd.DataFrame({"feature": features, "importance": base_model.feature_importances_}).sort_values("importance", ascending=False)
    importance.to_csv(REPORTS_DIR / "classifier_feature_importance.csv", index=False)

    predictions = test_df[["Order_ID", "lane", "Disruption_Event", "is_disrupted"]].copy()
    predictions["predicted_probability"] = test_prob
    predictions["predicted_label"] = (test_prob >= threshold).astype(int)
    predictions.to_csv(REPORTS_DIR / "classifier_final_test_predictions.csv", index=False)

    joblib.dump(base_model, MODELS_DIR / "disruption_classifier_base_xgb.pkl")
    if calibrated_model is not None:
        joblib.dump(calibrated_model, MODELS_DIR / "disruption_classifier_calibrated.pkl")
        joblib.dump(calibrated_model, MODELS_DIR / "disruption_classifier.pkl")
        saved_model = calibrated_model
    else:
        joblib.dump(base_model, MODELS_DIR / "disruption_classifier.pkl")
        saved_model = base_model

    print("\nClassifier candidate selection, validation only:")
    print(candidate_df.to_string(index=False))
    print("\nDisruption classifier final-test metrics:")
    printable = pd.DataFrame([final_metrics]).drop(columns=["validation_metrics_at_chosen_threshold", "baselines", "classifier_candidate_validation_summary"]).T.rename(columns={0: "value"})
    print(printable.to_string())
    print("\nDisruption classifier final-test classification report:")
    print(classification_report(y_test, (test_prob >= threshold).astype(int), zero_division=0))
    print("Top classifier features:")
    print(importance.head(15).to_string(index=False))
    return saved_model, base_model, final_metrics, lane_map, (train_idx, val_idx, test_idx)

# Active exception detector: separate post-departure operational detector

def train_active_exception_detector(data: pd.DataFrame, features: List[str]):
    y = data["is_active_exception"].copy()
    train_idx, val_idx, test_idx = split_indices_for_classification(y)
    train_df = data.iloc[train_idx].copy()
    val_df = data.iloc[val_idx].copy()
    test_df = data.iloc[test_idx].copy()
    X_train, y_train = train_df[features].copy(), train_df["is_active_exception"].copy()
    X_val, y_val = val_df[features].copy(), val_df["is_active_exception"].copy()
    X_test, y_test = test_df[features].copy(), test_df["is_active_exception"].copy()

    scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    model = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=3,
        gamma=0.5,
        scale_pos_weight=float(scale_pos),
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.05,
        reg_lambda=1.0,
        eval_metric="aucpr",
        tree_method="hist",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_prob = model.predict_proba(X_test)[:, 1]
    threshold = 0.50
    metrics = classification_metrics_dict(y_test, y_prob, threshold)
    metrics.update({
        "model_file": "active_exception_detector.pkl",
        "target": "is_active_exception",
        "mode": "post_departure_operational_detector",
        "feature_count": len(features),
        "threshold_tuned_on": None,
        "threshold_status": "fixed_0.50",
        "scale_pos_weight": float(scale_pos),
        "best_iteration": None if getattr(model, "best_iteration", None) is None else int(model.best_iteration),
        "split_train_rows": int(len(train_df)),
        "split_validation_rows": int(len(val_df)),
        "split_final_test_rows": int(len(test_df)),
        "train_positive_rate": float(y_train.mean()),
        "validation_positive_rate": float(y_val.mean()),
        "test_positive_rate": float(y_test.mean()),
        "baselines": classification_baselines(y_train, y_test),
    })

    importance = pd.DataFrame({"feature": features, "importance": model.feature_importances_}).sort_values("importance", ascending=False)
    importance.to_csv(REPORTS_DIR / "active_exception_feature_importance.csv", index=False)
    predictions = test_df[["Order_ID", "lane", "is_event_disruption", "is_late", "is_active_exception"]].copy()
    predictions["predicted_probability"] = y_prob
    predictions["predicted_label"] = (y_prob >= threshold).astype(int)
    predictions.to_csv(REPORTS_DIR / "active_exception_final_test_predictions.csv", index=False)
    joblib.dump(model, MODELS_DIR / "active_exception_detector.pkl")

    print("\nActive exception detector final-test metrics:")
    print(pd.DataFrame([metrics]).drop(columns=["baselines"]).T.rename(columns={0: "value"}).to_string())
    print("\nActive exception detector final-test classification report:")
    print(classification_report(y_test, (y_prob >= threshold).astype(int), zero_division=0))
    print("Top active exception features:")
    print(importance.head(15).to_string(index=False))
    return model, metrics

# Cost predictors: validation-only early stopping, final-test metrics, and cost baselines

def train_cost_predictor(data: pd.DataFrame, features: List[str], output_name: str, report_prefix: str):
    train_idx, val_idx, test_idx = split_indices_for_regression(len(data))
    train_df = data.iloc[train_idx].copy()
    val_df = data.iloc[val_idx].copy()
    test_df = data.iloc[test_idx].copy()
    X_train, y_train = train_df[features].copy(), train_df["Shipping_Cost_USD"].copy()
    X_val, y_val = val_df[features].copy(), val_df["Shipping_Cost_USD"].copy()
    X_test, y_test = test_df[features].copy(), test_df["Shipping_Cost_USD"].copy()

    model = XGBRegressor(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.80,
        reg_alpha=0.05,
        reg_lambda=1.0,
        tree_method="hist",
        early_stopping_rounds=50,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred = np.maximum(model.predict(X_test), 0)
    metrics = cost_metrics_dict(y_test, y_pred, y_train=y_train)
    metrics.update({
        "model_file": output_name,
        "feature_count": len(features),
        "target": "Shipping_Cost_USD",
        "split_train_rows": int(len(train_df)),
        "split_validation_rows": int(len(val_df)),
        "split_final_test_rows": int(len(test_df)),
        "best_iteration": None if getattr(model, "best_iteration", None) is None else int(model.best_iteration),
    })

    importance = pd.DataFrame({"feature": features, "importance": model.feature_importances_}).sort_values("importance", ascending=False)
    importance.to_csv(REPORTS_DIR / f"{report_prefix.lower()}_cost_feature_importance.csv", index=False)
    predictions = test_df[["Order_ID", "lane", "Shipping_Cost_USD"]].copy()
    predictions["predicted_shipping_cost_usd"] = y_pred
    predictions["absolute_error_usd"] = np.abs(predictions["Shipping_Cost_USD"] - predictions["predicted_shipping_cost_usd"])
    predictions.to_csv(REPORTS_DIR / f"{report_prefix.lower()}_cost_final_test_predictions.csv", index=False)
    joblib.dump(model, MODELS_DIR / output_name)

    print(f"\n{report_prefix} cost predictor final-test metrics:")
    print(pd.DataFrame([metrics]).T.rename(columns={0: "value"}).to_string())
    print(f"Top {report_prefix} cost features:")
    print(importance.head(15).to_string(index=False))
    return model, metrics

In [ ]:
# Cell 7: Train and save the models

classifier_model, classifier_base_model, classifier_metrics, lane_rate_map, classifier_split_indices = train_disruption_classifier(df, CLASSIFIER_FEATURES)
active_exception_model, active_exception_metrics = train_active_exception_detector(df, ACTIVE_EXCEPTION_FEATURES)
cost_model, cost_metrics = train_cost_predictor(
    df,
    COST_MODEL_FEATURES_AGENT_COMPATIBLE,
    output_name="cost_predictor.pkl",
    report_prefix="agent_compatible",
)
all_metrics = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_path": str(DATA_PATH),
    "row_count": int(len(df)),
    "version": "v2_clean_validation_final_test",
    "split_design": "60% train / 20% validation / 20% final test; final test is not used for early stopping, calibration, or threshold tuning",
    "use_leakage_safe_classifier": bool(USE_LEAKAGE_SAFE_CLASSIFIER),
    "use_smote": bool(USE_SMOTE),
    "calibrate_classifier": bool(CALIBRATE_CLASSIFIER),
    "calibration_method": CALIBRATION_METHOD if CALIBRATE_CLASSIFIER else None,
    "data_quality": data_quality_report,
    "disruption_classifier": classifier_metrics,
    "active_exception_detector": active_exception_metrics,
    "cost_predictor_agent_compatible": cost_metrics,
}
if TRAIN_ENHANCED_COST_MODEL:
    enhanced_cost_model, enhanced_cost_metrics = train_cost_predictor(
        df,
        COST_MODEL_FEATURES_ENHANCED,
        output_name="cost_predictor_enhanced.pkl",
        report_prefix="enhanced",
    )
    all_metrics["cost_predictor_enhanced"] = enhanced_cost_metrics

In [ ]:
# Cell 8: Build the text knowledge base used by the RAG layer

def build_knowledge_base(data: pd.DataFrame) -> List[str]:
    docs = []
    disrupted = data[data["is_disrupted"] == 1].copy()
    if not disrupted.empty:
        stats = (
            disrupted.groupby(["Disruption_Event", "Mitigation_Action_Taken"], dropna=False)
            .agg(
                count=("Order_ID", "count"),
                avg_delay=("observed_delay_days", "mean"),
                avg_cost=("Shipping_Cost_USD", "mean"),
                on_time_rate=("is_late", lambda x: 1 - x.mean()),
            )
            .reset_index()
            .sort_values(["Disruption_Event", "on_time_rate"], ascending=[True, False])
        )
        for _, row in stats.iterrows():
            docs.append(
                f"MITIGATION PLAYBOOK RECORD\n"
                f"Disruption Type      : {row['Disruption_Event']}\n"
                f"Mitigation Action    : {row['Mitigation_Action_Taken']}\n"
                f"Historical On-Time   : {row['on_time_rate']:.1%}\n"
                f"Avg Delay (days)     : {row['avg_delay']:.1f}\n"
                f"Avg Shipping Cost    : ${row['avg_cost']:,.0f}\n"
                f"Sample Size          : {int(row['count'])} orders"
            )
    route_stats = data.groupby("Route_Type").agg(
        avg_geo_risk=("Geopolitical_Risk_Index", "mean"),
        avg_weather=("Weather_Severity_Index", "mean"),
        disruption_rate=("is_disrupted", "mean"),
        avg_delay=("observed_delay_days", "mean"),
        avg_cost=("Shipping_Cost_USD", "mean"),
        order_count=("Order_ID", "count"),
    ).reset_index()
    for _, row in route_stats.iterrows():
        docs.append(
            f"ROUTE RISK PROFILE\n"
            f"Route                : {row['Route_Type']}\n"
            f"Avg Geopolitical Risk: {row['avg_geo_risk']:.3f}\n"
            f"Avg Weather Severity : {row['avg_weather']:.2f}\n"
            f"Disruption Rate      : {row['disruption_rate']:.1%}\n"
            f"Avg Delay (days)     : {row['avg_delay']:.1f}\n"
            f"Avg Shipping Cost    : ${row['avg_cost']:,.0f}\n"
            f"Total Orders         : {int(row['order_count'])}"
        )
    product_stats = data.groupby("Product_Category").agg(
        disruption_rate=("is_disrupted", "mean"),
        avg_delay=("observed_delay_days", "mean"),
        avg_cost=("Shipping_Cost_USD", "mean"),
        criticality=("product_criticality", "first"),
    ).reset_index()
    for _, row in product_stats.iterrows():
        docs.append(
            f"PRODUCT RISK PROFILE\n"
            f"Product Category     : {row['Product_Category']}\n"
            f"Criticality Score    : {row['criticality']:.2f}\n"
            f"Disruption Rate      : {row['disruption_rate']:.1%}\n"
            f"Avg Delay (days)     : {row['avg_delay']:.1f}\n"
            f"Avg Shipping Cost    : ${row['avg_cost']:,.0f}"
        )
    if not disrupted.empty and "lane" in disrupted.columns:
        lane_disruption = (
            disrupted.groupby(["lane", "Disruption_Event", "Mitigation_Action_Taken"], dropna=False)
            .agg(count=("Order_ID", "count"), avg_delay=("observed_delay_days", "mean"), avg_cost=("Shipping_Cost_USD", "mean"))
            .reset_index()
        )
        qualifying = lane_disruption[lane_disruption["count"] >= 3]
        if not qualifying.empty:
            best_by_lane = qualifying.loc[qualifying.groupby(["lane", "Disruption_Event"])["avg_delay"].idxmin()]
            for _, row in best_by_lane.iterrows():
                docs.append(
                    f"LANE-SPECIFIC MITIGATION RECORD\n"
                    f"Lane                 : {row['lane']}\n"
                    f"Disruption Type      : {row['Disruption_Event']}\n"
                    f"Best Action          : {row['Mitigation_Action_Taken']}\n"
                    f"Avg Delay (days)     : {row['avg_delay']:.1f}\n"
                    f"Avg Shipping Cost    : ${row['avg_cost']:,.0f}\n"
                    f"Sample Size          : {int(row['count'])} orders"
                )
    seasonal = (
        data.groupby(["Route_Type", "order_month"])
        .agg(disruption_rate=("is_disrupted", "mean"), order_count=("Order_ID", "count"), avg_delay=("observed_delay_days", "mean"))
        .reset_index()
    )
    risky_seasonal = seasonal[(seasonal["disruption_rate"] > 0.15) & (seasonal["order_count"] >= 20)].sort_values("disruption_rate", ascending=False).head(20)
    for _, row in risky_seasonal.iterrows():
        docs.append(
            f"SEASONAL RISK RECORD\n"
            f"Route                : {row['Route_Type']}\n"
            f"Order Month          : {int(row['order_month'])}\n"
            f"Disruption Rate      : {row['disruption_rate']:.1%}\n"
            f"Avg Delay (days)     : {row['avg_delay']:.1f}\n"
            f"Sample Size          : {int(row['order_count'])} orders"
        )
    playbook_path = KB_DIR / "mitigation_playbook.txt"
    playbook_path.write_text("\n\n---\n\n".join(docs), encoding="utf-8")
    return docs


def cost_reference_stats(frame: pd.DataFrame) -> Dict[str, object]:
    s = frame["Shipping_Cost_USD"].dropna().astype(float)
    if s.empty:
        return {}
    q = s.quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    return {
        "min": float(s.min()),
        "p10": float(q.loc[0.10]),
        "p25": float(q.loc[0.25]),
        "p50": float(q.loc[0.50]),
        "p75": float(q.loc[0.75]),
        "p90": float(q.loc[0.90]),
        "max": float(s.max()),
        "mean": float(s.mean()),
        "std": float(s.std(ddof=0)),
        "count": int(s.shape[0]),
    }


def write_json(path: Path, obj) -> None:
    def default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, (np.ndarray,)):
            return o.tolist()
        if isinstance(o, (pd.Series, pd.Index)):
            return o.tolist()
        if pd.isna(o) if not isinstance(o, (dict, list, tuple, str, bytes)) else False:
            return None
        raise TypeError(f"Object of type {type(o)} is not JSON serializable")
    path.write_text(json.dumps(obj, indent=2, default=default), encoding="utf-8")

In [ ]:
# Cell 9: Save feature configuration and metrics

kb_docs = build_knowledge_base(df)
route_cost_reference_stats = {str(route): cost_reference_stats(group) for route, group in df.groupby("Route_Type", dropna=False)}
route_cost_reference_stats["__global__"] = cost_reference_stats(df)

feature_config = {
    "version": "v2_clean_validation_final_test",
    "disruption_probability_threshold": classifier_metrics["threshold"],
    "use_leakage_safe_classifier": USE_LEAKAGE_SAFE_CLASSIFIER,
    "use_smote": USE_SMOTE,
    "threshold_optimized": OPTIMIZE_THRESHOLD,
    "threshold_tuned_on": "validation",
    "threshold_min_precision": THRESHOLD_MIN_PRECISION,
    "threshold_min_recall": THRESHOLD_MIN_RECALL,
    "threshold_max_alert_rate": THRESHOLD_MAX_ALERT_RATE,
    "calibrate_classifier": CALIBRATE_CLASSIFIER,
    "calibration_method": CALIBRATION_METHOD if CALIBRATE_CLASSIFIER else None,
    "classifier_features_active": CLASSIFIER_FEATURES,
    "classifier_features_demo_agent_compatible": CLASSIFIER_FEATURES_DEMO,
    "classifier_features_leakage_safe": CLASSIFIER_FEATURES_LEAKAGE_SAFE,
    "active_exception_features": ACTIVE_EXCEPTION_FEATURES,
    "cost_model_features_agent_compatible": COST_MODEL_FEATURES_AGENT_COMPATIBLE,
    "cost_model_features_enhanced": COST_MODEL_FEATURES_ENHANCED,
    "temporal_features": TEMPORAL_FEATURES,
    "lane_features": LANE_FEATURES,
    "route_risk_map": ROUTE_RISK_MAP,
    "product_criticality_map": PRODUCT_CRITICALITY_MAP,
    "disruption_label_map": DISRUPTION_LABEL_MAP,
    "route_cost_reference_stats": route_cost_reference_stats,
    "lane_disruption_rate_map": lane_rate_map,
    "lane_disruption_rate_source": "training_split_only_no_final_test_leakage",
}

write_json(MODELS_DIR / "feature_config.json", feature_config)
write_json(REPORTS_DIR / "metrics.json", all_metrics)
write_json(REPORTS_DIR / "data_quality_report.json", data_quality_report)

summary_rows = []
summary_rows.append({"model": "disruption_classifier", **{k: classifier_metrics[k] for k in ["roc_auc", "pr_auc", "precision", "recall", "f1", "accuracy", "alert_rate", "brier_score"]}})
summary_rows.append({"model": "active_exception_detector", **{k: active_exception_metrics[k] for k in ["roc_auc", "pr_auc", "precision", "recall", "f1", "accuracy", "alert_rate", "brier_score"]}})
summary_rows.append({"model": "cost_predictor_agent_compatible", "mae": cost_metrics["mae"], "median_absolute_error": cost_metrics["median_absolute_error"], "wape_pct": cost_metrics["wape_pct"], "rmsle": cost_metrics["rmsle"], "r2": cost_metrics["r2"]})
if TRAIN_ENHANCED_COST_MODEL:
    summary_rows.append({"model": "cost_predictor_enhanced", "mae": enhanced_cost_metrics["mae"], "median_absolute_error": enhanced_cost_metrics["median_absolute_error"], "wape_pct": enhanced_cost_metrics["wape_pct"], "rmsle": enhanced_cost_metrics["rmsle"], "r2": enhanced_cost_metrics["r2"]})
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(REPORTS_DIR / "metrics_summary.csv", index=False)
print("\nMetrics summary:")
print(summary_df.to_string(index=False))

# Smoke test saved models.
sample = df.iloc[0].copy()
sample_lane_rate = lane_rate_map.get(str(sample["lane"]), lane_rate_map["__global__"])
sample["lane_disruption_rate"] = sample_lane_rate
loaded_classifier_path = MODELS_DIR / "disruption_classifier.pkl"
loaded_classifier = joblib.load(loaded_classifier_path)
loaded_cost_model = joblib.load(MODELS_DIR / "cost_predictor.pkl")
classifier_vector = pd.DataFrame([sample[CLASSIFIER_FEATURES].astype(float).to_dict()], columns=CLASSIFIER_FEATURES)
cost_vector = pd.DataFrame([sample[COST_MODEL_FEATURES_AGENT_COMPATIBLE].astype(float).to_dict()], columns=COST_MODEL_FEATURES_AGENT_COMPATIBLE)
prob = float(loaded_classifier.predict_proba(classifier_vector)[0, 1])
cost = float(loaded_cost_model.predict(cost_vector)[0])
print("\nSmoke test:")
print(f"Sample Order_ID: {sample['Order_ID']}")
print(f"Lane disruption rate used: {sample_lane_rate:.4f}")
print(f"Predicted disruption probability: {prob:.4f}")
print(f"Predicted shipping cost: ${cost:,.2f}")
if TRAIN_ENHANCED_COST_MODEL:
    loaded_enhanced_cost_model = joblib.load(MODELS_DIR / "cost_predictor_enhanced.pkl")
    enhanced_vector = pd.DataFrame([sample[COST_MODEL_FEATURES_ENHANCED].astype(float).to_dict()], columns=COST_MODEL_FEATURES_ENHANCED)
    enhanced_cost = float(loaded_enhanced_cost_model.predict(enhanced_vector)[0])
    print(f"Enhanced cost model prediction: ${enhanced_cost:,.2f}")

zip_base = WORKING_DIR / "supply_chain_model_artifacts"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)
print("\nSaved artifacts:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(OUTPUT_DIR))
print(f"\nCreated artifact ZIP: {zip_path}")
print(f"Knowledge-base documents written: {len(kb_docs)}")

In [ ]:
# Cell 10: Smoke test note

# The saved-model smoke test now runs in Cell 9 after all artifacts are written.
# It loads disruption_classifier.pkl, cost_predictor.pkl, and cost_predictor_enhanced.pkl
# from disk to verify artifact compatibility with the agent runtime.


In [ ]:
# Cell 11: Artifact bundle note

# Cell 9 creates supply_chain_model_artifacts.zip in the working directory.
# In Kaggle, download this ZIP from the notebook Output panel.


## FAISS index note

This notebook intentionally does **not** build the FAISS vector index. It only creates `knowledge_base/mitigation_playbook.txt`. After downloading the artifacts and placing the GenAI agent beside them, build the RAG index with:

```bash
python supply_chain_genai_agent_groq_hf.py index
```

That command creates:

```text
supply_chain_agent_training_outputs/knowledge_base/faiss_index/index.faiss
supply_chain_agent_training_outputs/knowledge_base/faiss_index/index.pkl
```

## Using the generated files with your GenAI agent

After running this notebook, download `supply_chain_model_artifacts.zip` and extract it beside the GenAI agent file:

```text
project_folder/
  supply_chain_genai_agent_groq_hf.py
  supply_chain_agent_training_outputs/
    models/
      disruption_classifier.pkl
      cost_predictor.pkl
      cost_predictor_enhanced.pkl
      feature_config.json
    knowledge_base/
      mitigation_playbook.txt
```

Then run:

```bash
python supply_chain_genai_agent_groq_hf.py check
python supply_chain_genai_agent_groq_hf.py index
```

Use `--replay-mode` only when replaying historical CSV rows where `Disruption_Event` should be treated as a known alert:

```bash
python supply_chain_genai_agent_groq_hf.py resolve --order-idx 10 --replay-mode
```

For real-time JSON inputs, omit `--replay-mode` so `Disruption_Event` cannot bypass the classifier.